# Square-QDM checkerboard-family evidence for Sec. VII

This notebook implements the gated fixed-width workflow in `SEC7_NUMERICAL_PROVISIONING.md`.
It first tests the energy-density, transport, and reduced-sector gates.  The expensive
family-wide thermal scan runs only after the structural gates pass.  If the nonzero
uniform potential fails the $\beta=0$ energy-density gate, the automatic protocol uses
an energy-matched finite-$\beta$ comparator and records that the main-text $\beta=0$
claim is unavailable.

## Imports and run controls

In [ ]:
from dataclasses import replace
from itertools import product
from pathlib import Path
import sys, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la
import scipy.sparse as sp
from scipy.optimize import brentq
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate qlinks repository root")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from helpers import (
    PRX_FOUR_PANEL_FIGSIZE, add_panel_label, orthonormalize_columns,
    projector_deleted_block_covariance, projector_resolved_energy_basis,
    save_prx_figure, set_revtex_matplotlib_style, use_integer_ticks,
    write_figure_manifest,
)
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    CageClassificationConfig, CageSearchConfig, CageSearcher, LocalQDMCageSearchConfig, RobustQDMLocalCageSearchConfig, LocalWitnessTemplate,
    SquareQDMPeriodicProductUnitCell, SquareQDMWitnessPlacement,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_instance,
    certify_square_qdm_periodic_product_sequence,
    classify_cage_state, commuting_cyclic_symmetry_sector_basis, diagnose_eigenpair,
    directed_transition_witness_template, evaluate_square_qdm_classification_witnesses_on_strips,
    local_witnesses_from_classification_report, materialize_square_qdm_periodic_product_state,
    project_operator_to_sector, project_state_to_sector,
    robust_qdm_local_cage_search, scan_square_qdm_beta_zero_energy_density,
    select_microcanonical_window_by_width, thermodynamic_energy_window_plan,
)
from qlinks.models import SquareQDMModel, qdm_plaquette_link_gauge_matrix
from qlinks.models.couplings import peierls_plaquette_coupling

TOL=1e-10
RANK_TOL=1e-9
DARK_TOL=1e-9
RUN_PROFILE='smoke'
STRICT_CLAIMS=False
TRANSPORT_REPEATS_BY_PROFILE={'smoke':(1,2,3),'known':(1,2,3),'production':(1,2,3)}
ED_REPEATS_BY_PROFILE={'smoke':(1,), 'known':(1,2), 'production':(1,2)}
TRANSFER_MAX_LENGTH_BY_PROFILE={'smoke':32,'known':128,'production':256}
CHECKERBOARD_TRANSPORT_REPEATS=TRANSPORT_REPEATS_BY_PROFILE[RUN_PROFILE]
CHECKERBOARD_ED_REPEATS=ED_REPEATS_BY_PROFILE[RUN_PROFILE]
CHECKERBOARD_TRANSFER_MAX_LENGTH=TRANSFER_MAX_LENGTH_BY_PROFILE[RUN_PROFILE]
CHECKERBOARD_PHASE_VALUES=(0.0,0.025,0.05,0.075,0.10)
CHECKERBOARD_POSITIVE_PHASE_VALUES=(0.025,0.05,0.075,0.10)
CHECKERBOARD_REPRESENTATIVE_PHASE=0.05
CHECKERBOARD_THERMAL_PROTOCOL='auto'  # auto, beta0, finite-beta
CHECKERBOARD_ENERGY_MATCH_TOL=1e-3
MICROCANONICAL_PREFACTORS=(0.50,0.75,1.00)
PRIMARY_WINDOW_PREFACTOR=0.75
RUN_CHECKERBOARD_THERMAL_SCAN=True
RUN_CHECKERBOARD_CONCENTRATION=True
RUN_LARGE_STRIP=False
LARGE_STRIP_REPEATS=(3,)
ENERGY_BLOCK_TOL=1e-9
USE_TEX=False
SAVE_FIGURES=True
SAVE_PDF=True
FIGURE_FORMATS=('pdf','svg')
DATA_DIR=REPO_ROOT/'experimental'/'data'/'square_qdm_draft_evidence'
FIGURE_DIR=DATA_DIR/'figures'
DATA_DIR.mkdir(parents=True,exist_ok=True); FIGURE_DIR.mkdir(parents=True,exist_ok=True)
set_revtex_matplotlib_style(base_font_size=9.0,prefer_tex=USE_TEX)

def save_figure(fig,stem):
    if SAVE_FIGURES:
        save_prx_figure(fig,stem,directory=FIGURE_DIR,formats=FIGURE_FORMATS)

def project_operator_to_sector_sparse(operator, sector):
    basis=sector.basis if hasattr(sector,'basis') else sector
    return sp.csr_array(basis.conj().T @ (operator @ basis))

def translation_permutation(model, configs, *, dx=0, dy=0):
    lookup={}
    for link in model.lattice.links:
        x,y=model.lattice.sites[int(link.source)].cell
        lookup[(int(x),int(y),str(link.kind))]=int(link.id)
    transformed=np.zeros_like(configs)
    for link in model.lattice.links:
        x,y=model.lattice.sites[int(link.source)].cell
        target=lookup[((int(x)+dx)%model.lx,(int(y)+dy)%model.ly,str(link.kind))]
        transformed[:,target]=configs[:,int(link.id)]
    index={tuple(map(int,row)):i for i,row in enumerate(configs)}
    return np.asarray([index[tuple(map(int,row))] for row in transformed],dtype=np.int64)

print({'profile':RUN_PROFILE,'transport_repeats':CHECKERBOARD_TRANSPORT_REPEATS,
       'ed_repeats':CHECKERBOARD_ED_REPEATS,'phase_values':CHECKERBOARD_PHASE_VALUES})

## Recover the repeated compact cage and bounded kinetic witnesses

In [ ]:
base_model=SquareQDMModel(lx=4,ly=4,boundary_condition='periodic',winding_x=0,winding_y=0,
    winding_convention='electric',coup_kin=1.0,coup_pot=1.0)
local_cfg=LocalQDMCageSearchConfig(halo_layers=0,boundary_mode='relaxed',prune_inactive_local_basis_states=True,
    tolerance=TOL,degenerate_basis_strategy='ipr',ipr_random_seed=1234)
robust_cfg=RobustQDMLocalCageSearchConfig(local_config=local_cfg,region_strategies=('stripe',),stripe_widths=(1,),
    stripe_directions=(0,1),max_regions_per_strategy=None,block_signatures=((0,2),),max_records_per_region=2,
    min_blocks=2,max_blocks=None,max_product_support_size=2048,max_paddings_per_stage=100,
    max_paddings_per_packing=10,include_sectors=True,padding_stages=('static',),tolerance=1e-9,store_full_states=False)
stripe_certified,stripe_context=robust_qdm_local_cage_search(base_model,config=robust_cfg,return_context=True)
repeatable=[]
for report_index,report in enumerate(stripe_certified.reports):
    try:
        cell=SquareQDMPeriodicProductUnitCell.from_padding(base_model,stripe_context.blocks,report.padding,repeat_axis='x')
        certificate=certify_square_qdm_periodic_product_sequence(cell)
    except ValueError:
        continue
    if certificate.is_certified:
        repeatable.append((report_index,cell,certificate))
if not repeatable: raise RuntimeError('No repeatable compact cage found')
repeatable_report_index,product_unit_cell,product_sequence=repeatable[0]
stripe_record=stripe_certified.records[repeatable_report_index]
classification=classify_cage_state(stripe_record.cage_state,kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy='infer_support_component'))
strip_report=evaluate_square_qdm_classification_witnesses_on_strips(classification,model=base_model,lengths=(4,8,12),
    winding_sector=(0,0),normalization='operator_norm',winding_projection='fourier')
z_reference=strip_report.records[0].witness
z_placement=strip_report.records[0].placement
op=np.asarray(z_reference.template.local_operator,dtype=np.complex128)
adj=np.abs(op)>TOL; target=int(np.argmax(np.sum(adj,axis=1))); sources=np.flatnonzero(adj[target])
a_template=directed_transition_witness_template(target_pattern=z_reference.template.local_patterns[target],
    source_patterns=[z_reference.template.local_patterns[i] for i in sources],
    amplitudes=[op[target,i] for i in sources],metadata={'name':'A_R'},normalization='operator_norm')
a_reference=a_template.instantiate(z_reference.variable_indices)
a_placement=SquareQDMWitnessPlacement.from_local_witness(base_model,a_reference)
a_cert=certify_local_witness_on_square_qdm_periodic_sequence(product_sequence,a_reference)
z_cert=certify_local_witness_on_square_qdm_periodic_sequence(product_sequence,z_reference)
pd.DataFrame([{'witness':'A','annihilation_residual':a_cert.annihilation_residual,'Q_norm':a_cert.witness.q_operator_norm},
              {'witness':'Z','annihilation_residual':z_cert.annihilation_residual,'Q_norm':z_cert.witness.q_operator_norm}]).to_csv(
              DATA_DIR/'qdm_checkerboard_AZ_certificates.csv',index=False)
print(product_sequence.to_summary_dict())

## Gate 1: fixed-width energy-density matching

In [ ]:
transfer_lengths=tuple(range(4,CHECKERBOARD_TRANSFER_MAX_LENGTH+1,4))
energy_scan=scan_square_qdm_beta_zero_energy_density([(L,4) for L in transfer_lengths],potential_coupling=1.0,
    winding_sector=(0,0),winding_projection='fourier')
cage_ed=float(product_sequence.energy_density)
energy_rows=[]
for ev in energy_scan.evaluations:
    d=ev.to_summary_dict()
    energy_rows.append({'Lx':int(ev.length),'Ly':4,'cage_energy_density':cage_ed,
        'beta0_trace_energy_density':float(ev.energy_density),
        'signed_mismatch':float(ev.energy_density-cage_ed),
        'absolute_mismatch':abs(float(ev.energy_density-cage_ed)),
        'partition_method':'exact_transfer_fourier'})
energy_match=pd.DataFrame(energy_rows)
energy_match.to_csv(DATA_DIR/'qdm_checkerboard_energy_density_match.csv',index=False)
fit_rows=[]
fit_frame=energy_match[energy_match['Lx']>=max(12,transfer_lengths[len(transfer_lengths)//4])]
L=fit_frame['Lx'].to_numpy(float); y=fit_frame['signed_mismatch'].to_numpy(float)
for name,x in [('constant',np.zeros_like(L)),('Delta_inf+c/Lx',1/L),('Delta_inf+c/Lx^2',1/L**2)]:
    if name=='constant': intercept=float(np.mean(y)); slope=0.; pred=np.full_like(y,intercept)
    else: slope,intercept=np.polyfit(x,y,1); pred=intercept+slope*x
    fit_rows.append({'fit_form':name,'included_Lx':','.join(map(str,L.astype(int))),
        'limit':float(intercept),'slope':float(slope),'rmse':float(np.sqrt(np.mean((y-pred)**2)))})
energy_fit=pd.DataFrame(fit_rows)
energy_fit['gate_tolerance']=CHECKERBOARD_ENERGY_MATCH_TOL
energy_fit.to_csv(DATA_DIR/'qdm_checkerboard_energy_density_fit.csv',index=False)
preferred=float(energy_fit.loc[energy_fit.fit_form=='Delta_inf+c/Lx','limit'].iloc[0])
BETA0_GATE_PASSED=abs(preferred)<=CHECKERBOARD_ENERGY_MATCH_TOL
ACTIVE_THERMAL_PROTOCOL=('beta0' if BETA0_GATE_PASSED else 'finite-beta') if CHECKERBOARD_THERMAL_PROTOCOL=='auto' else CHECKERBOARD_THERMAL_PROTOCOL
pd.DataFrame([{'gate':'fixed_width_energy_density','passed':BETA0_GATE_PASSED,'preferred_limit':preferred,
    'tolerance':CHECKERBOARD_ENERGY_MATCH_TOL,'selected_protocol':ACTIVE_THERMAL_PROTOCOL}]).to_csv(
    DATA_DIR/'qdm_checkerboard_gate_status.csv',index=False)
display(energy_fit); print({'beta0_gate_passed':BETA0_GATE_PASSED,'active_protocol':ACTIVE_THERMAL_PROTOCOL})

## Gate 2: checkerboard transport, local constraints, gauge quotient, and common sector

In [ ]:
def checkerboard_sign(model,pid):
    x,y=model.lattice.plaquette_anchor_cell(int(pid)); return 1 if (int(x)+int(y))%2==0 else -1

def checkerboard_instance(repeats,phase):
    raw=product_unit_cell.instantiate(int(repeats))
    model0=replace(raw.model,winding_x=0,winding_y=0)
    couplings={int(pid):peierls_plaquette_coupling(1.0,float(phase)*checkerboard_sign(model0,pid))
               for pid in model0.plaquette_ids()}
    model=replace(model0,coup_kin=couplings,coup_pot=1.0)
    return replace(raw,model=model)

family_rows=[]; constraint_rows=[]; gauge_rows=[]; sector_rows=[]
for repeats in CHECKERBOARD_TRANSPORT_REPEATS:
    Lx=4*int(repeats)
    for phase in CHECKERBOARD_PHASE_VALUES:
        instance=checkerboard_instance(repeats,phase)
        cert=certify_square_qdm_periodic_product_instance(instance,tolerance=1e-9)
        family_rows.append({'repeats':repeats,'Lx':Lx,'Ly':4,'phase':phase,'cage_residual':max([v for b in cert.block_certificates for v in (b.kinetic_residual,b.potential_residual,b.leakage_residual)],default=0.0),
            'cage_energy':float(cert.energy.real),'cage_energy_density':float(cert.energy.real/(4*Lx)),
            'winding_sector':repr(cert.winding_sector),'A_localized_residual':a_cert.annihilation_residual,
            'Z_localized_residual':z_cert.annihilation_residual,'local_certificate_passed':cert.is_certified})
    model=checkerboard_instance(repeats,0.0).model
    active=set(int(pid) for block in checkerboard_instance(repeats,0.0).blocks for pid in block.record.active_plaquette_ids)
    for x in range(Lx):
        for y in range(2):
            p1=int(model.lattice.plaquette_id_from_cell(x,y)); p2=int(model.lattice.plaquette_id_from_cell(x,(y+2)%4))
            chi1=checkerboard_sign(model,p1); chi2=checkerboard_sign(model,p2)
            constraint_rows.append({'repeats':repeats,'Lx':Lx,'row_x':x,'row_y':y,'plaquette_1':p1,'plaquette_2':p2,
                'chi_1':chi1,'chi_2':chi2,'equal_phase_constraint_passed':chi1==chi2,
                'both_active':p1 in active and p2 in active})
    chi=np.asarray([checkerboard_sign(model,pid) for pid in model.plaquette_ids()],float)
    incidence=qdm_plaquette_link_gauge_matrix(model.lattice)
    proj=incidence@np.linalg.lstsq(incidence,chi,rcond=RANK_TOL)[0]
    gauge_rows.append({'repeats':repeats,'Lx':Lx,'plaquette_count':len(chi),
        'link_gauge_rank':int(np.linalg.matrix_rank(incidence,tol=RANK_TOL)),
        'checkerboard_norm':float(np.linalg.norm(chi)),'distance_from_link_gauge_image':float(np.linalg.norm(chi-proj)),
        'relative_distance':float(np.linalg.norm(chi-proj)/np.linalg.norm(chi))})
    if repeats in CHECKERBOARD_ED_REPEATS:
        instance=checkerboard_instance(repeats,CHECKERBOARD_REPRESENTATIVE_PHASE)
        build=instance.model.build(basis_solver='dfs',builder='bitmask',backend='scipy',sort_basis=True)
        configs=basis_configs_from_build_result(build); cage=materialize_square_qdm_periodic_product_state(instance,configs)
        tx2=translation_permutation(instance.model,configs,dx=2); ty2=translation_permutation(instance.model,configs,dy=2)
        sector=commuting_cyclic_symmetry_sector_basis((tx2,ty2),orders=(Lx//2,2),momentum_indices=(0,0),
            labels={'Tx2_k':0,'Ty2_k':0})
        projected=project_state_to_sector(cage,sector); pnorm=float(np.linalg.norm(projected))
        if pnorm>TOL: projected/=pnorm
        qvals={}
        for name,placement in [('A',a_placement),('Z',z_placement)]:
            op=placement.instantiate_on_model(instance.model).embed(configs)
            q=project_operator_to_sector(op.conj().T@op,sector)
            qvals[name]=float(np.vdot(projected,q@projected).real)
        sector_rows.append({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'Tx2_order':Lx//2,'Ty2_order':2,'kx2_index':0,'ky2_index':0,
            'sector_dimension':sector.sector_dimension,'cage_projection_norm':pnorm,
            'projected_QA':qvals['A'],'projected_QZ':qvals['Z'],'status':'verified_ED'})
    else:
        sector_rows.append({'repeats':repeats,'Lx':Lx,'phase':CHECKERBOARD_REPRESENTATIVE_PHASE,
            'Tx2_order':Lx//2,'Ty2_order':2,'kx2_index':0,'ky2_index':0,
            'sector_dimension':np.nan,'cage_projection_norm':np.nan,'projected_QA':np.nan,'projected_QZ':np.nan,
            'status':'local_transport_verified_sector_projection_pending_large_strip'})
family=pd.DataFrame(family_rows); constraints=pd.DataFrame(constraint_rows); gauge=pd.DataFrame(gauge_rows); common_sector=pd.DataFrame(sector_rows)
family.to_csv(DATA_DIR/'qdm_checkerboard_fixed_width_family.csv',index=False)
constraints.to_csv(DATA_DIR/'qdm_checkerboard_compatibility_constraints.csv',index=False)
gauge.to_csv(DATA_DIR/'qdm_checkerboard_gauge_quotient.csv',index=False)
common_sector.to_csv(DATA_DIR/'qdm_checkerboard_common_symmetry_sector.csv',index=False)
GATE2_LOCAL_PASSED=bool(family.local_certificate_passed.all() and constraints.equal_phase_constraint_passed.all())
display(family); display(common_sector)

### Reference Type-I continuation inventory

In [ ]:
reference_build=base_model.build(basis_solver='dfs',builder='sparse',backend='scipy',sort_basis=True)
reference_search=CageSearcher.from_model_build_result(reference_build,config=CageSearchConfig(
    search_type='type1',tolerance=TOL,degenerate_basis_strategy='ipr',ipr_n_restarts=32,
    ipr_candidate_count=24,ipr_random_seed=1234)).run()
type1_rows=[]
for signature in reference_search.signatures:
    for record_index,record in enumerate(reference_search[signature]):
        state=np.zeros(reference_search.hilbert_size,dtype=np.complex128)
        state[np.asarray(record.cage_state.support,dtype=np.int64)]=record.cage_state.local_state
        for phase in CHECKERBOARD_PHASE_VALUES:
            model=checkerboard_instance(1,phase).model
            build=model.build(basis_solver='dfs',builder='sparse',backend='scipy',sort_basis=True)
            np.testing.assert_array_equal(build.basis.states,reference_build.basis.states)
            report=diagnose_eigenpair(build.hamiltonian,state)
            type1_rows.append({'signature':repr(signature),'record_index':record_index,'support_size':len(record.cage_state.support),
                'phase':phase,'energy_real':float(report.energy.real),'energy_imag':float(report.energy.imag),
                'residual':float(report.residual_norm),'status':'exact' if report.residual_norm<=1e-8 else 'lifted',
                'inventory_scope':'4x4 reference Type-I states continued along checkerboard path'})
type1=pd.DataFrame(type1_rows)
type1.to_csv(DATA_DIR/'qdm_checkerboard_type1_continuation.csv',index=False)
display(type1.groupby(['phase','status']).size().rename('count').reset_index())

## Complete stripe-local algebra and translated joint-dark machinery

In [ ]:
def stripe_algebra(configs,model,sector):
    witness=z_placement.instantiate_on_model(model); variables=tuple(map(int,witness.variable_indices))
    patterns=np.unique(configs[:,variables],axis=0)
    position={v:i for i,v in enumerate(variables)}
    site_lookup={tuple(site.cell):int(site.id) for site in model.lattice.sites}
    boundary=[]
    for cell in z_placement.affected_sites:
        sid=site_lookup[tuple(cell)]
        inc=[int(model.layout.link_variable_index(link)) for link in model.lattice.incident_links(sid)]
        if not all(v in position for v in inc): boundary.append(tuple(inc))
    signatures=[]
    for pattern in patterns:
        signatures.append(tuple(sum(int(pattern[position[v]]) for v in inc if v in position) for inc in boundary))
    groups={}
    for i,sig in enumerate(signatures): groups.setdefault(sig,[]).append(i)
    matrices=[]; names=[]
    n=len(patterns)
    for block_id,indices in enumerate(groups.values()):
        for i in indices:
            m=np.zeros((n,n),complex); m[i,i]=1; matrices.append(m); names.append(f'b{block_id}_d{i}')
        for pos,i in enumerate(indices):
            for j in indices[pos+1:]:
                m=np.zeros((n,n),complex); m[i,j]=m[j,i]=1/np.sqrt(2); matrices.append(m); names.append(f'b{block_id}_s{i}_{j}')
                m=np.zeros((n,n),complex); m[i,j]=-1j/np.sqrt(2); m[j,i]=1j/np.sqrt(2); matrices.append(m); names.append(f'b{block_id}_a{i}_{j}')
    projected=[]; kept=[]
    for name,matrix in zip(names,matrices,strict=True):
        template=LocalWitnessTemplate(pattern_key=(),local_patterns=tuple(tuple(map(int,p)) for p in patterns),
            local_operator=matrix,metadata={'name':name,'boundary_flux_blocks':len(groups)})
        full=template.instantiate(variables).embed(configs)
        op=project_operator_to_sector_sparse(full,sector)
        norm=float(np.sqrt(np.real(op.conj().multiply(op).sum())))
        if norm>1e-12: projected.append(op/norm); kept.append(name)
    return projected,kept,{'local_pattern_count':len(patterns),'boundary_flux_block_count':len(groups),
        'boundary_flux_block_dimensions':repr(sorted(map(len,groups.values()))),'formal_operator_dimension':sum(len(v)**2 for v in groups.values()),
        'projected_operator_dimension':len(projected)}

def translated_joint_dark(configs,model,sector):
    total=None
    for x in range(model.lx):
        for placement in (a_placement,z_placement):
            op=placement.instantiate_on_model(model,origin_x=x).embed(configs)
            q=op.conj().T@op
            total=q if total is None else total+q
    return project_operator_to_sector(total,sector)

def joint_dark_kernel(energies,vectors,q_all,tower):
    groups=[]
    for i,e in enumerate(energies):
        if not groups or abs(e-energies[groups[-1][-1]])>ENERGY_BLOCK_TOL: groups.append([i])
        else: groups[-1].append(i)
    cols=[]; rows=[]
    for bid,g in enumerate(groups):
        basis=vectors[:,g]; comp=basis.conj().T@(q_all@basis); comp=0.5*(comp+comp.conj().T)
        vals,rot=la.eigh(comp,check_finite=False); scale=max(1.,float(np.max(np.abs(vals),initial=0.)))
        keep=np.flatnonzero(vals<=DARK_TOL*scale); dark=basis@rot[:,keep] if keep.size else np.zeros((basis.shape[0],0),complex)
        weight=float(np.linalg.norm(dark.conj().T@tower)**2) if keep.size else 0.
        cols.extend(dark[:,i] for i in range(dark.shape[1]))
        rows.append({'energy_block_id':bid,'energy':float(np.mean(energies[g])),'block_dimension':len(g),
            'joint_dark_rank':int(keep.size),'target_weight':weight,'remaining_rank_after_target':max(0,int(keep.size)-(weight>1-1e-7))})
    exceptional=orthonormalize_columns(np.column_stack(cols),tolerance=1e-9) if cols else np.zeros((vectors.shape[0],0),complex)
    return exceptional,rows

def canonical_weights(energies,target):
    energies=np.asarray(energies,float)
    def weights(beta):
        x=-beta*energies; x-=np.max(x); w=np.exp(x); return w/w.sum()
    f0=float(np.mean(energies)-target)
    if abs(f0)<1e-12:return 0.,weights(0.)
    direction=1. if f0>0 else -1.; bound=direction
    while abs(bound)<128 and (np.dot(weights(bound),energies)-target)*f0>0: bound*=2
    if abs(bound)>=128: raise RuntimeError('Could not bracket finite-beta match')
    beta=float(brentq(lambda b:np.dot(weights(b),energies)-target,*sorted((0.,bound))))
    return beta,weights(beta)

## Gate 3 and energy-resolved family pilot

In [ ]:
thermal_rows=[]; beta0_rows=[]; scatter_frames=[]; dark_rows=[]; concentration_rows=[]; trace_rows=[]; cleaning_rows=[]
if RUN_CHECKERBOARD_THERMAL_SCAN and GATE2_LOCAL_PASSED:
  for repeats in CHECKERBOARD_ED_REPEATS:
    for phase in CHECKERBOARD_PHASE_VALUES:
      instance=checkerboard_instance(repeats,phase); model=instance.model; Lx=model.lx; volume=Lx*4
      build=model.build(basis_solver='dfs',builder='bitmask',backend='scipy',sort_basis=True); configs=basis_configs_from_build_result(build)
      cage=materialize_square_qdm_periodic_product_state(instance,configs)
      tx2=translation_permutation(model,configs,dx=2); ty2=translation_permutation(model,configs,dy=2)
      sector=commuting_cyclic_symmetry_sector_basis((tx2,ty2),orders=(Lx//2,2),momentum_indices=(0,0))
      tower=project_state_to_sector(cage,sector); tower/=np.linalg.norm(tower)
      h=project_operator_to_sector(build.hamiltonian,sector); energies,vectors=la.eigh(h,check_finite=False)
      local={name:placement.instantiate_on_model(model).embed(configs) for name,placement in [('A',a_placement),('Z',z_placement)]}
      q={name:project_operator_to_sector(op.conj().T@op,sector) for name,op in local.items()}
      raw_Q={name:np.real(np.einsum('ij,ij->j',vectors.conj(),op@vectors)) for name,op in q.items()}
      q_all=translated_joint_dark(configs,model,sector); exceptional,jrows=joint_dark_kernel(energies,vectors,q_all,tower)
      for r in jrows:r.update({'repeats':repeats,'Lx':Lx,'phase':phase})
      dark_rows.extend(jrows)
      resolved=projector_resolved_energy_basis(energies,vectors,exceptional,energy_tolerance=ENERGY_BLOCK_TOL,vector_tolerance=1e-9)
      keep=~resolved['is_exceptional'].astype(bool); clean_E=resolved['energies'][keep]
      clean_Q={name:np.real(np.einsum('ij,ij->j',resolved['basis'][:,keep].conj(),op@resolved['basis'][:,keep])) for name,op in q.items()}
      target=float(np.vdot(tower,h@tower).real)
      beta0_weights=np.full(clean_E.size,1/clean_E.size)
      raw_beta0_weights=np.full(energies.size,1/energies.size)
      beta0_reference={name:float(np.dot(beta0_weights,vals)) for name,vals in clean_Q.items()}
      beta0_reference_raw={name:float(np.dot(raw_beta0_weights,vals)) for name,vals in raw_Q.items()}
      if ACTIVE_THERMAL_PROTOCOL=='beta0':
          beta=0.; reference_weights=beta0_weights; beta_raw=0.; reference_weights_raw=raw_beta0_weights
      else:
          beta,reference_weights=canonical_weights(clean_E,target)
          beta_raw,reference_weights_raw=canonical_weights(energies,target)
      reference={name:float(np.dot(reference_weights,vals)) for name,vals in clean_Q.items()}
      reference_raw={name:float(np.dot(reference_weights_raw,vals)) for name,vals in raw_Q.items()}
      trace_rows.append({'repeats':repeats,'Lx':Lx,'phase':phase,'sector_dimension':sector.sector_dimension,
        'beta0_energy_density_clean':float(np.dot(beta0_weights,clean_E)/volume),'target_energy_density':target/volume,
        'tau_A_beta0_clean':float(np.dot(beta0_weights,clean_Q['A'])),'tau_Z_beta0_clean':float(np.dot(beta0_weights,clean_Q['Z'])),
        'tau_A_transfer':np.nan,'tau_Z_transfer':np.nan})
      primary_indices=None
      for pref in MICROCANONICAL_PREFACTORS:
        plan=thermodynamic_energy_window_plan(volume=volume,energy_density=target/volume,width_prefactor=pref,local_energy_scale=1.)
        window=select_microcanonical_window_by_width(clean_E,target_energy=target,half_width=plan.half_width,degeneracy_tolerance=ENERGY_BLOCK_TOL)
        idx=np.asarray(window.indices,int); mc={name:float(np.mean(vals[idx])) for name,vals in clean_Q.items()}
        raw_window=select_microcanonical_window_by_width(energies,target_energy=target,half_width=plan.half_width,degeneracy_tolerance=ENERGY_BLOCK_TOL)
        raw_idx=np.asarray(raw_window.indices,int); mc_raw={name:float(np.mean(vals[raw_idx])) for name,vals in raw_Q.items()}
        row={'repeats':repeats,'Lx':Lx,'Ly':4,'phase':phase,'thermal_protocol':ACTIVE_THERMAL_PROTOCOL,'matched_beta':beta,
          'sector_dimension':sector.sector_dimension,'cage_energy':target,'cage_energy_density':target/volume,
          'cage_residual':float(np.linalg.norm(h@tower-target*tower)),'window_prefactor':pref,'window_half_width':window.half_width,
          'window_energy_density_half_width':window.half_width/volume,'window_state_count':window.n_states,
          'joint_dark_rank':exceptional.shape[1],'removed_fraction':float(exceptional.shape[1]/max(1,raw_window.n_states)),
          'raw_window_state_count':raw_window.n_states,'clean_window_state_count':window.n_states,
          'tau_A_mc':mc['A'],'tau_Z_mc':mc['Z'],'tau_A_mc_raw':mc_raw['A'],'tau_Z_mc_raw':mc_raw['Z'],
          'tau_A_reference':reference['A'],'tau_Z_reference':reference['Z'],
          'tau_A_reference_raw':reference_raw['A'],'tau_Z_reference_raw':reference_raw['Z'],'matched_beta_raw':beta_raw,
          'delta_A':abs(mc['A']-reference['A']),'delta_Z':abs(mc['Z']-reference['Z']),
          'Delta':max(abs(mc['A']-reference['A']),abs(mc['Z']-reference['Z'])),
          'cage_QA':float(np.vdot(tower,q['A']@tower).real),'cage_QZ':float(np.vdot(tower,q['Z']@tower).real)}
        thermal_rows.append(row)
        beta0_row=dict(row)
        beta0_row.update({'thermal_protocol':'beta0_diagnostic' if BETA0_GATE_PASSED else 'beta0_gate_failed_diagnostic',
            'matched_beta':0.0,'matched_beta_raw':0.0,
            'tau_A_reference':beta0_reference['A'],'tau_Z_reference':beta0_reference['Z'],
            'tau_A_reference_raw':beta0_reference_raw['A'],'tau_Z_reference_raw':beta0_reference_raw['Z'],
            'delta_A':abs(mc['A']-beta0_reference['A']),'delta_Z':abs(mc['Z']-beta0_reference['Z']),
            'Delta':max(abs(mc['A']-beta0_reference['A']),abs(mc['Z']-beta0_reference['Z']))})
        beta0_rows.append(beta0_row)
        cleaning_rows.append({'repeats':repeats,'Lx':Lx,'phase':phase,'window_prefactor':pref,
            'raw_window_state_count':raw_window.n_states,'clean_window_state_count':window.n_states,
            'removed_joint_dark_rank':exceptional.shape[1],'removed_fraction':float(exceptional.shape[1]/max(1,raw_window.n_states)),
            'tau_A_mc_raw':mc_raw['A'],'tau_A_mc_clean':mc['A'],'tau_Z_mc_raw':mc_raw['Z'],'tau_Z_mc_clean':mc['Z'],
            'tau_A_reference_raw':reference_raw['A'],'tau_A_reference_clean':reference['A'],
            'tau_Z_reference_raw':reference_raw['Z'],'tau_Z_reference_clean':reference['Z'],
            'energy_block_tolerance':ENERGY_BLOCK_TOL,'dark_tolerance':DARK_TOL})
        if abs(pref-PRIMARY_WINDOW_PREFACTOR)<TOL: primary_indices=idx
      if primary_indices is not None:
        scatter=pd.DataFrame({'repeats':repeats,'Lx':Lx,'phase':phase,'energy':clean_E,
            'energy_density':clean_E/volume,'Q_A':clean_Q['A'],'Q_Z':clean_Q['Z'],'is_tower_state':False})
        scatter_frames.append(scatter)
        if RUN_CHECKERBOARD_CONCENTRATION:
          ops,names,meta=stripe_algebra(configs,model,sector)
          covariance=projector_deleted_block_covariance(energies,vectors,exceptional,ops,
              np.asarray(select_microcanonical_window_by_width(energies,target_energy=target,
              half_width=thermodynamic_energy_window_plan(volume=volume,energy_density=target/volume,
              width_prefactor=PRIMARY_WINDOW_PREFACTOR,local_energy_scale=1.).half_width,
              degeneracy_tolerance=ENERGY_BLOCK_TOL).indices,int),energy_tolerance=ENERGY_BLOCK_TOL,vector_tolerance=1e-9)
          concentration_rows.append({'repeats':repeats,'Lx':Lx,'phase':phase,'operator_space_dimension':len(ops),
              **meta,'largest_covariance_eigenvalue':covariance['largest_eigenvalue'],'w':covariance['largest_width'],
              'median_nonidentity_width':covariance['median_nonidentity_width'],'energy_block_tolerance':ENERGY_BLOCK_TOL,
              'worst_coefficients':repr(dict(zip(names,np.asarray(covariance['worst_coefficients']).tolist(),strict=True)))})
thermal=pd.DataFrame(thermal_rows); beta0=pd.DataFrame(beta0_rows); scatter=pd.concat(scatter_frames,ignore_index=True) if scatter_frames else pd.DataFrame()
dark=pd.DataFrame(dark_rows); concentration=pd.DataFrame(concentration_rows); traces=pd.DataFrame(trace_rows); cleaning=pd.DataFrame(cleaning_rows)
beta0.to_csv(DATA_DIR/'qdm_checkerboard_beta0_overlap.csv',index=False)
thermal.to_csv(DATA_DIR/'qdm_checkerboard_window_systematics.csv',index=False)
dark.to_csv(DATA_DIR/'qdm_checkerboard_joint_dark_kernel.csv',index=False)
cleaning.to_csv(DATA_DIR/'qdm_checkerboard_cleaning_audit.csv',index=False)
thermal.to_csv(DATA_DIR/'qdm_checkerboard_thermal_overlap.csv',index=False)
concentration.to_csv(DATA_DIR/'qdm_checkerboard_concentration_grid.csv',index=False)
concentration.to_csv(DATA_DIR/'qdm_checkerboard_worst_eigenoperator.csv',index=False)
traces.to_csv(DATA_DIR/'qdm_checkerboard_resolved_beta0_trace.csv',index=False)
if not scatter.empty:
    rep=scatter[np.isclose(scatter.phase,CHECKERBOARD_REPRESENTATIVE_PHASE)]
    Lmax=int(rep.Lx.max()); rep[rep.Lx==Lmax].to_csv(DATA_DIR/'qdm_checkerboard_eth_scatter.csv',index=False)
# Gate-3 transfer comparison for A/Z.
from qlinks.caging import SquareQDMStripTransferMatrix
transfer=SquareQDMStripTransferMatrix(circumference=4); overlap_rows=[]
for name,placement in [('A',a_placement),('Z',z_placement)]:
    scaling=transfer.scan_witness(placement,lengths=tuple(4*r for r in CHECKERBOARD_ED_REPEATS),boundary_x='periodic',
        winding_sector=(0,0),winding_projection='fourier')
    for ev in scaling.evaluations: overlap_rows.append({'witness':name,'Lx':ev.length,'transfer_target':ev.expectation})
transfer_targets=pd.DataFrame(overlap_rows)
if not traces.empty:
    for i,row in traces.iterrows():
        for name in ('A','Z'):
            match=transfer_targets[(transfer_targets.witness==name)&(transfer_targets.Lx==row.Lx)]
            if len(match): traces.loc[i,f'tau_{name}_transfer']=float(match.transfer_target.iloc[0])
for name in ('A','Z'):
    traces[f'delta_{name}_resolved_transfer']=np.abs(traces[f'tau_{name}_beta0_clean']-traces[f'tau_{name}_transfer']) if not traces.empty else np.nan
traces.to_csv(DATA_DIR/'qdm_checkerboard_resolved_beta0_trace.csv',index=False)
traces.to_csv(DATA_DIR/'qdm_checkerboard_transfer_sector_overlap.csv',index=False)
# Diagnostic fit files; never call two lengths a thermodynamic extrapolation.
fit_rows=[]
primary=thermal[np.isclose(thermal.window_prefactor,PRIMARY_WINDOW_PREFACTOR)] if not thermal.empty else thermal
for phase,group in primary.groupby('phase') if not primary.empty else []:
    fit_rows.append({'phase':phase,'included_Lx':','.join(map(str,sorted(group.Lx.unique()))),
        'status':'insufficient_lengths' if group.Lx.nunique()<3 else 'fit_provisioned'})
pd.DataFrame(fit_rows).to_csv(DATA_DIR/'qdm_checkerboard_matching_distance_fit.csv',index=False)
pd.DataFrame(fit_rows).to_csv(DATA_DIR/'qdm_checkerboard_fixed_width_shared_fit.csv',index=False)
pd.DataFrame(fit_rows).to_csv(DATA_DIR/'qdm_checkerboard_uniform_concentration_fit.csv',index=False)
display(thermal[thermal.window_prefactor==PRIMARY_WINDOW_PREFACTOR] if not thermal.empty else thermal)

## Select a provisional pilot phase and render the reserved Fig. 7

The phase remains provisional until Gates 1--3 and the non-target joint-dark audit pass at the production lengths.


In [ ]:
primary=thermal[np.isclose(thermal.window_prefactor,PRIMARY_WINDOW_PREFACTOR)].copy() if not thermal.empty else pd.DataFrame()
if not primary.empty:
    candidates=primary[primary.phase.isin(CHECKERBOARD_POSITIVE_PHASE_VALUES)]
    available=sorted(candidates.phase.unique())
    phi_star=CHECKERBOARD_REPRESENTATIVE_PHASE if CHECKERBOARD_REPRESENTATIVE_PHASE in available else available[len(available)//2]
    dark_at_phi=dark[np.isclose(dark.phase,phi_star)] if not dark.empty else pd.DataFrame()
    remaining_dark=int(dark_at_phi.remaining_rank_after_target.max()) if not dark_at_phi.empty else -1
    conc_at_phi=concentration[np.isclose(concentration.phase,phi_star)] if not concentration.empty else pd.DataFrame()
    conc_width=float(conc_at_phi.w.max()) if not conc_at_phi.empty else np.nan
    delta_at_phi=float(candidates[np.isclose(candidates.phase,phi_star)].Delta.max())
    selection_status='provisional_gates_pending' if remaining_dark<=0 else 'provisional_extra_joint_dark_present'
    pd.DataFrame([{'phi_star':phi_star,'selection':selection_status,'thermal_protocol':ACTIVE_THERMAL_PROTOCOL,
        'maximum_residual_non_target_dark_rank':remaining_dark,'maximum_sampled_Delta':delta_at_phi,
        'maximum_sampled_concentration_width':conc_width}]).to_csv(
        DATA_DIR/'qdm_checkerboard_representative_phase.csv',index=False)
    # Data-only notebook preview; final renderer uses the same contract.
    fig=plt.figure(figsize=PRX_FOUR_PANEL_FIGSIZE); gs=fig.add_gridspec(2,2,wspace=.32,hspace=.36)
    axes=[fig.add_subplot(gs[i,j]) for i in range(2) for j in range(2)]
    ax=axes[0]; scat=pd.read_csv(DATA_DIR/'qdm_checkerboard_eth_scatter.csv')
    for col,label,marker in [('Q_A',r'$Q_R^A$','o'),('Q_Z',r'$Q_R^Z$','s')]:
        ax.scatter(scat.energy_density,scat[col],s=10,alpha=.5,marker=marker,label=label)
    tower_row=primary[(primary.Lx==primary[primary.phase==phi_star].Lx.max())&np.isclose(primary.phase,phi_star)].iloc[0]
    ax.scatter([tower_row.cage_energy_density],[0],marker='*',s=75,edgecolors='black',linewidths=.4,zorder=6,label='cage')
    ax.set_xlabel(r'$e=E/(4L_x)$'); ax.set_ylabel('Witness activity'); ax.grid(alpha=.22); add_panel_label(ax,'(a)')
    ax=axes[1]; rep=primary[np.isclose(primary.phase,phi_star)].sort_values('Lx')
    for key,label,marker in [('A',r'$Q_R^A$','o'),('Z',r'$Q_R^Z$','s')]:
        ax.plot(rep.Lx,rep[f'tau_{key}_mc'],marker=marker,label=label+' MC')
        ax.plot(rep.Lx,rep[f'tau_{key}_reference'],marker=marker,fillstyle='none',ls='--',label=label+' reference')
    use_integer_ticks(ax,axis='x'); ax.set_xticks(rep.Lx.astype(int)); ax.set_xlabel(r'$L_x$'); ax.set_ylabel(r'$\tau$'); ax.grid(alpha=.22); add_panel_label(ax,'(b)')
    ax=axes[2]
    for Lx,g in primary[primary.phase.isin(CHECKERBOARD_POSITIVE_PHASE_VALUES)].groupby('Lx'):
        ax.plot(g.phase,g.Delta,marker='o',label=rf'$L_x={int(Lx)}$')
    ax.set_xlabel(r'$\varphi$'); ax.set_ylabel(r'$\Delta_{L_x}(\varphi)$'); ax.grid(alpha=.22); add_panel_label(ax,'(c)')
    ax=axes[3]
    if not concentration.empty:
        c=concentration[concentration.phase.isin(CHECKERBOARD_POSITIVE_PHASE_VALUES)]
        piv=c.pivot(index='Lx',columns='phase',values='w').sort_index()
        x=np.asarray(piv.columns,float); y=np.asarray(piv.index,int)
        xe=np.r_[x[0]-(x[1]-x[0])/2,(x[:-1]+x[1:])/2,x[-1]+(x[-1]-x[-2])/2] if len(x)>1 else np.array([x[0]-.0125,x[0]+.0125])
        ye=np.r_[y[0]-2,(y[:-1]+y[1:])/2,y[-1]+2] if len(y)>1 else np.array([y[0]-1,y[0]+1])
        mesh=ax.pcolormesh(xe,ye,piv.to_numpy(),shading='flat'); fig.colorbar(mesh,ax=ax,label=r'$w_{L_x}(\varphi)$')
        ax.set_yticks(y); use_integer_ticks(ax,axis='y')
    ax.set_xlabel(r'$\varphi$'); ax.set_ylabel(r'$L_x$'); add_panel_label(ax,'(d)')
    handles,labels=axes[0].get_legend_handles_labels(); fig.legend(handles,labels,loc='upper center',ncol=3,bbox_to_anchor=(.5,.99))
    save_figure(fig,'qdm_checkerboard_figure7_combined'); plt.show()
else:
    print('Thermal scan unavailable; Fig. 7 remains provisioned.')

## Output and claim manifest

In [ ]:
gate_rows=[
 {'gate':'1_energy_density','status':'passed' if BETA0_GATE_PASSED else 'failed_beta0_switch_finite_beta','detail':f'limit={preferred:.6g}'},
 {'gate':'2_local_transport','status':'passed' if GATE2_LOCAL_PASSED else 'failed','detail':'4x4,8x4,12x4 local certificates'},
 {'gate':'2_common_sector','status':'finite_ED_verified_plus_large_strip_pending','detail':'T_x^2,T_y^2 sector verified on enumerated lengths; 12x4 projection pending'},
 {'gate':'3_resolved_transfer','status':'finite_size_only','detail':'ED lengths only; third length pending'},]
pd.DataFrame(gate_rows).to_csv(DATA_DIR/'qdm_checkerboard_scientific_gates.csv',index=False)
pd.DataFrame([{'claim_id':'checkerboard_family','status':'candidate_sampled_compatibility',
 'thermal_protocol':ACTIVE_THERMAL_PROTOCOL,'third_energy_resolved_length_available':False,
 'note':'Do not claim fixed-width ICQMBS until a third length and controlled concentration extrapolation are available.'}]).to_csv(DATA_DIR/'claim_manifest.csv',index=False)
write_figure_manifest(DATA_DIR/'figure_manifest.json')
for path in sorted(DATA_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(DATA_DIR))